![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 01 | Framing and Baseline

*Project 3 — AI Product: From Model to Product*

**Team:**
**Track:** model / LLM
**Data:**

---

### Data Source

Job listings were collected using the Adzuna API, focusing on Frankfurt and surrounding areas within a 50 km radius.

The dataset contains 162 unique job advertisements across software engineering, machine learning, data analysis, cloud engineering, and technical support.

Each row represents a job advertisement with its title, company, location, description, original URL, and search keyword.

The downloaded dataset is stored locally in `data/raw/jobs.csv`.

In [4]:
# %pip install sentence-transformers

---
## 0. Setup

In [5]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# from src.functions import *
import json
from pathlib import Path
from IPython.display import display, clear_output
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")
from sklearn.feature_extraction.text import TfidfVectorizer


---
## 1. Frame the product

Before any code. In a few sentences:

- **The user:** who uses your product, and in what situation?
- **The input and the output:** what do they give it, and what do they get back?
- **The decision:** what do they do with the output?
- **The metric:** how will you know it works, and which mistake costs more? (*Model track:* a missed complaint or a false alarm? *LLM track:* a wrong answer, an answer with no source, or a refusal?)

You wrote a first version on launch day. Refine it here.


The Problem

Finding relevant job vacancies can be time-consuming.

Traditional keyword-based job searches often:
- Miss relevant jobs that use different terminology.
- Return positions requiring different skills.
- Provide limited personalization based on a candidate's experience.

### Our Solution

JobMatch AI recommends vacancies based on a candidate's
skills and professional experience.

Users can:
- Upload a PDF CV or describe their experience.
- Specify their preferred location.
- Choose remote, hybrid, or on-site work.
- Receive their top 5–10 matching job vacancies.

### Our Research Question

**Can semantic matching improve job recommendations
compared with traditional keyword-based matching?**

---
## 2. Load the data

Run `python download_data.py` (or `python download_data.py clothing`) from the repo root first. Your files land in `data/raw/`, which is gitignored.

In [6]:
df = pd.read_csv("../data/raw/jobs.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (162, 7)


,id,title,company,location,description,url,search_keyword
0,5407298685,Python Developer,Global Market Solutions Germany,"Hausen, Frankfurt am Main","GMS is an operational consultancy with expertise in data and digital systems. Whether digitalisation, data & analytics or compliance - our consultants are the missing piece of our customer’s digit...",https://www.adzuna.de/details/5407298685?utm_medium=api&utm_source=87077f8e,Python
1,5931600841,Python Entwickler (m/w/d),Instaffo GmbH,"Eschborn, Main-Taunus-Kreis",Du bist interessiert an der Stelle als Python Entwickler (m/w/d) bei just.nerds GmbH & Co. KG ? Auf Instaffo kannst du dich für diesen und weitere Jobs ohne großen Aufwand bewerben. Erstelle dein ...,https://www.adzuna.de/details/5931600841?utm_medium=api&utm_source=87077f8e,Python
2,5938275963,Consultant - Python (m/w/d),x-markets consulting,"Eschborn, Main-Taunus-Kreis","Die Boutique-Beratung für den KapitalmarktSeit nunmehr 25 Jahren gestaltet x-markets consulting als kleine, feine Beratungsboutique den Kapitalmarkt mit seinen hochwertigen Leistungen für namhafte...",https://www.adzuna.de/details/5938275963?utm_medium=api&utm_source=87077f8e,Python
3,5533884834,Senior FullStack Python & JS Developer -DEU,Cryoport Systems,"Darmstadt, Hessen","Cryoport Systems is a comprehensive supply chain partner for the life sciences industry, delivering specialized solutions to meet the challenges of the biopharmaceutical, cell and gene therapy, re...",https://www.adzuna.de/details/5533884834?utm_medium=api&utm_source=87077f8e,Python
4,5940743171,Automotive Test Automation Engineer – Python & AI,Da Vinci Engineering,"Eschborn, Main-Taunus-Kreis","Stellen-ID: 22567 Zweck und Ziel der Stelle We are searching intelligent and innovative employees. If you are interested in working for a dynamic company with flat hierarchy, we look forward to re...",https://www.adzuna.de/details/5940743171?utm_medium=api&utm_source=87077f8e,Python


---
## 3. First look

Shape, columns, a handful of rows **read in full**. What is one row? For the Amazon data, join a few reviews to their product in the metadata file on `parent_asin`.

In [7]:
df.isnull().sum()

id                0
title             0
company           1
location          0
description       0
url               0
search_keyword    0
dtype: int64

---
## 4. Data quality

Empty and very short texts, HTML (`<br />`, `&#34;`), duplicates, texts in other languages, missing titles or metadata. Write down what you find and what you will do about it. Put the cleaning in a function in `src/functions.py`: the app will need it too.

In [8]:
print(df["description"].str.len().describe())

print("\nExample job description:\n")
print(df["description"].iloc[0])


count    162.000000
mean     497.944444
std       22.053647
min      225.000000
25%      500.000000
50%      500.000000
75%      500.000000
max      500.000000
Name: description, dtype: float64

Example job description:

GMS is an operational consultancy with expertise in data and digital systems. Whether digitalisation, data & analytics or compliance - our consultants are the missing piece of our customer’s digital puzzle. We empower their ongoing digital transformation and transform challenges into opportunities. We're part of the MoOngy Group, a more than 18 years old European network of companies with a focus on IT & Engineering expertise. Your tasks as Python developer: Bachelor's degree in Computer Scienc…


---
## 5. Look at the texts

Lengths, the most frequent words, how the texts differ between ratings, products or periods. Read some hard cases: the 3-star reviews, the sarcastic ones, the ones whose rating and text disagree.

In [9]:
print("Unique job titles:", df["title"].nunique())
print("Unique companies:", df["company"].nunique())

display(df["search_keyword"].value_counts())

Unique job titles: 131
Unique companies: 97


search_keyword
Python                           20
Data Analyst                     20
Data Scientist                   19
Tableau                          19
Software engineers               19
Cloud engineers                  19
Machine Learning                 18
Backend Developer                14
Computer programmer               7
Technical support specialists     7
Name: count, dtype: int64

---
## 6. Model track — the target and the split

Skip this section on the LLM track.

- **Define the target** with a stated rule, for example stars to sentiment, and show the share of each class.
- **Split once**: a fixed `RANDOM_STATE`, stratified. Decide whether the split is random, by product (`parent_asin`) or by time, and say why. The test set is not touched again until notebook 03.

In [10]:
candidate_text = """
Python developer with experience in machine learning,
data analysis, SQL, Pandas, scikit-learn, and REST APIs.
Experience developing AI applications, working with
LLMs, and deploying Python applications.
Looking for a role in data science or AI engineering.
"""

candidate_profiles = {
    "ml_engineer": """
        Junior machine learning engineer with Python, Pandas,
        scikit-learn, SQL, data preprocessing, model evaluation,
        and experience building predictive models.
        Looking for a junior AI or machine learning position.
    """,

    "backend_developer": """
        Backend developer with Java, Spring Boot, REST APIs,
        SQL databases, unit testing, and microservices.
        Interested in backend software engineering roles.
    """,

    "data_analyst": """
        Data analyst experienced in SQL, Excel, Tableau,
        Power BI, dashboards, statistics, and reporting.
        Looking for business intelligence or data analyst roles.
    """,

    "cloud_engineer": """
        Cloud engineer with Microsoft Azure, Docker,
        Kubernetes, CI/CD, cloud infrastructure,
        and deployment automation experience.
    """,

    "technical_support": """
        IT support specialist with experience troubleshooting
        hardware and software issues, Windows administration,
        networking, help desk, and customer support.
    """
}

In [11]:
output = Path("../data/candidate_profiles.json")

with open(output, "w", encoding="utf-8") as f:
    json.dump(candidate_profiles, f, indent=4)

print(f"Saved {len(candidate_profiles)} candidate profiles")

Saved 5 candidate profiles


In [12]:
df["job_text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("")
)

In [13]:
# Combine candidate profile and job descriptions
documents = [candidate_text] + df["job_text"].tolist()

# Convert text into TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words="english")

tfidf_matrix = vectorizer.fit_transform(documents)

# Compare candidate against all jobs
similarities = cosine_similarity(
    tfidf_matrix[0:1],
    tfidf_matrix[1:]
).flatten()

df["tfidf_score"] = similarities

In [14]:
top_jobs = df.sort_values(
    "tfidf_score", ascending=False
).head(5)

display(
    top_jobs[
        ["title", "company", "location", "tfidf_score"]
    ]
)

,title,company,location,tfidf_score
17,Senior Cloud Developer Python | AI (m/w/d) 100.000€,SThree Germany,"Innenstadt, Aschaffenburg",0.163360
13,Cloud & AI Engineer Python (m/w/d) 100.000€,SThree Germany,"Sonnenberg, Wiesbaden",0.153385
14,Python Cloud Engineer - AI (m/w/d) 100.000€,SThree Germany,"Arheilgen, Darmstadt",0.153385
15,Cloud Software Engineer Python & AI (m/w/d) 100.000€,SThree Germany,"Altstadt, Mainz",0.150913
18,Cloud Software Engineer (m/w/d) Python & AI 100.000€,SThree Germany,"Niederrad, Frankfurt am Main",0.150913


In [15]:
feature_names = vectorizer.get_feature_names_out()

candidate_weights = tfidf_matrix[0].toarray().flatten()

word_importance = pd.DataFrame({
    "word": feature_names,
    "tfidf_weight": candidate_weights
})

display(
    word_importance
    .sort_values("tfidf_weight", ascending=False)
    .head(15)
)

,word,tfidf_weight
858,experience,0.408866
164,applications,0.378204
1887,python,0.227533
95,ai,0.212046
593,deploying,0.204433
618,developing,0.204433
1458,learn,0.204433
1487,llms,0.204433
1745,pandas,0.204433
1949,rest,0.204433


---
## 8. The baseline

The simplest thing that could work, scored with the same evaluation as your final solution.

- **Model track:** a `DummyClassifier`, then TF-IDF with logistic regression, scored with cross-validation on the training set.
- **LLM track:** the LLM alone (no retrieval, no tools) on your evaluation set, or a keyword search (TF-IDF) for the retrieval part. Log the tokens and the cost.

In [16]:
# # Select 30 jobs, using a fixed random seed
# evaluation_jobs = df.sample(
#     n=30,
#     random_state=42
# ).copy()

# # Keep columns needed for evaluation
# evaluation_jobs = evaluation_jobs[
#     ["id", "title", "description", "search_keyword"]
# ]

# print("Evaluation jobs:", len(evaluation_jobs))
# display(evaluation_jobs[["id", "title", "search_keyword"]])

In [17]:
evaluation_jobs = pd.read_csv(
    "../data/evaluation_jobs.csv",
    dtype={"id": str}
)

print("Evaluation jobs:", len(evaluation_jobs))

Evaluation jobs: 28


In [18]:
evaluation_jobs["search_keyword"].value_counts()

search_keyword
Data Analyst                     5
Machine Learning                 4
Software engineers               4
Python                           3
Cloud engineers                  3
Technical support specialists    3
Data Scientist                   2
Computer programmer              2
Backend Developer                1
Tableau                          1
Name: count, dtype: int64

In [19]:
display(
    evaluation_jobs[
        ["title", "search_keyword"]
    ].sort_values("search_keyword")
)

,title,search_keyword
7,Backend Developer (m/w/d) – IMES Industrie-Software / Microservices,Backend Developer
20,Cloud Engineer / Cloud Administrator (w/m/d),Cloud engineers
21,Azure Cloud Engineer (w/m/d),Cloud engineers
22,Cloud Operations Engineer (all genders) in Frankfurt,Cloud engineers
26,IT Support Mitarbeiter (m/w/d),Computer programmer
23,IT Support Mitarbeiter (m/w/d),Computer programmer
13,Data Analyst - Wettbewerbsanalyse (m/w/d),Data Analyst
14,Junior Data Analyst / Mitarbeiter Digitalisierung m/w/d,Data Analyst
12,Data Analyst als Senior Consultant SAP Analytics Cloud & Power BI (m/w/d),Data Analyst
17,"Deine Chance: Datenmanagement Upskilling - SQL, Microsoft Azure & Power BI für den schnellen Wiedereinstieg",Data Analyst


### Evaluation Design

To compare the matching approaches, I created five fictional candidate profiles representing different technical backgrounds.

A fixed evaluation set of 28 jobs was selected from the larger dataset using role-related keyword patterns and a reproducible random seed (`random_state=42`).

Each candidate was evaluated against the same 28 jobs, producing 140 candidate–job pairs.

Relevance was assessed on a three-level scale:

- **0:** Not relevant
- **1:** Partially relevant
- **2:** Highly relevant

The final labels were reviewed with AI assistance. The evaluation is therefore exploratory, and the results cannot be treated as independently verified ground truth.

The primary evaluation metrics are Precision@5 and nDCG@5.

In [20]:
role_patterns = {
    "ml_engineer": r"machine learning|data scientist|ai engineer|ki.engineer",
    "backend_developer": r"backend|java|spring|software engineer|software developer",
    "data_analyst": r"data analyst|business intelligence|bi |reporting|tableau",
    "cloud_engineer": r"cloud|devops|azure|platform engineer",
    "technical_support": r"it support|helpdesk|service desk|technical support|it administrator|systemadministrator"
}

for role, pattern in role_patterns.items():
    matches = df[
        df["title"].str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
    ]

    print(f"\n{role}: {len(matches)} matching job titles")
    display(matches[["id", "title"]].head(10))


ml_engineer: 17 matching job titles


,id,title
13,5953648768,Cloud & AI Engineer Python (m/w/d) 100.000€
20,5943389711,Data Scientist (m/w/d)
21,5953934080,Associate Data Scientist (m/w/d)
22,5062250361,Senior Data Scientist AI (m/w/d)
23,5936729063,Senior Data Scientist - Fokus Pharma (m/w/d)
25,5956672032,"(Junior) Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur"
26,5956672304,"Senior Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur"
27,5936199583,Machine Learning Engineer / Data Scientist Google Cloud (all genders)
30,5955434494,"Data Scientist für Compliance (m/w/d) - Datenbankentwicklung/BI, Risk / Compliance Management, Ingenieur"
40,5828492029,(Junior) Machine Learning Engineer (m/w/d)



backend_developer: 42 matching job titles


,id,title
7,5959431069,Backend-Entwickler (Python / PHP) (m/w/d)
8,5956294685,Backend-Entwickler (Python / PHP) (m/w/d)
9,5701642655,Backend-Entwickler (Python / PHP) (m/w/d)
10,5947098925,(Senior) Software Engineer – Java & Python (m/w/d)
11,5957955314,"Softwareentwickler Java und Python (w|m|d) - Anwendungsentwicklung, Ingenieur"
15,5953648780,Cloud Software Engineer Python & AI (m/w/d) 100.000€
18,5953648791,Cloud Software Engineer (m/w/d) Python & AI 100.000€
19,5956465101,(Senior) Data Engineer / Software Engineer – Python & Cloud (m/w/d)
44,5938631492,"Machine Learning Engineer (m/w/d) | Java, Python - 75.000€"
53,5907971202,Senior AI Software Engineer (all genders) in Frankfurt



data_analyst: 16 matching job titles


,id,title
71,5574048686,Data Analyst als Senior Consultant SAP Analytics Cloud & Power BI (m/w/d)
72,5943666507,Data Analyst - Wettbewerbsanalyse (m/w/d)
73,5950177911,Data Analyst - Marketing & Growth (w/m/d)
74,5950183749,Data Analyst - Chatbot & Voicebot (w/m/d)
75,5947325839,Business Process & Data Analyst bAV (m/w/d)
76,5945924160,Junior Data Analyst / Mitarbeiter Digitalisierung m/w/d
77,5956999493,Junior Data Analyst / Mitarbeiter Digitalisierung m/w/d
78,5751150763,Data Analyst / Power BI Professional (m/w/d)
79,5957378074,Trainee Financial Data Analyst – Wertpapierverwahrung (m/w/d)
80,5946147716,Specialist (m/w/d) AFC Performance & Reporting



cloud_engineer: 37 matching job titles


,id,title
5,5953191243,Netzwerkarchitekt - Cloud / Automation / CCNP (m/w/d)
6,5958035099,Senior Platform Engineer – Cloud & Analytics (m/w/d)
12,5889330917,DevOps Engineer (m/w/d)
13,5953648768,Cloud & AI Engineer Python (m/w/d) 100.000€
14,5953648802,Python Cloud Engineer - AI (m/w/d) 100.000€
15,5953648780,Cloud Software Engineer Python & AI (m/w/d) 100.000€
16,5953648782,AI Cloud Platform Engineer Python (m/w/d) 100.000€
17,5953648779,Senior Cloud Developer Python | AI (m/w/d) 100.000€
18,5953648791,Cloud Software Engineer (m/w/d) Python & AI 100.000€
19,5956465101,(Senior) Data Engineer / Software Engineer – Python & Cloud (m/w/d)



technical_support: 5 matching job titles


,id,title
91,5956294735,IT Support Mitarbeiter (m/w/d)
92,5959431067,IT Support Mitarbeiter (m/w/d)
117,5866031244,Aftersales Technical Support Specialist (m/w/d)
119,5959436878,"Werde IT Support Specialist (m/w/d), Quereinsteiger willkommen!"
120,5953993048,Remote Sensing Scientific and Technical Support - Passive Microwave Imaging Specialist (m/f/d) [26185]


In [21]:
selected_jobs = []

for role, pattern in role_patterns.items():

    matches = df[
        df["title"].str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
    ]

    # Select up to 6 jobs per category
    sample = matches.sample(
        n=min(6, len(matches)),
        random_state=42
    )

    selected_jobs.append(sample)

# Combine samples from all categories
evaluation_jobs = pd.concat(
    selected_jobs,
    ignore_index=True
)

# Remove jobs selected for multiple categories
evaluation_jobs = evaluation_jobs.drop_duplicates(
    subset=["id"]
).reset_index(drop=True)

print("Evaluation jobs:", len(evaluation_jobs))

display(
    evaluation_jobs[
        ["id", "title", "search_keyword"]
    ]
)

Evaluation jobs: 28


,id,title,search_keyword
0,5953648768,Cloud & AI Engineer Python (m/w/d) 100.000€,Python
1,5943389711,Data Scientist (m/w/d),Data Scientist
2,5956672032,"(Junior) Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur",Data Scientist
3,5939347586,AI Engineer (m/w/d) - Implementation & Consulting,Machine Learning
4,5892995304,Computer Vision Intern – Machine Learning Operations (m/f/d),Machine Learning
5,5727342997,Senior Machine Learning Engineer – Time Series & Forecasting (w/m/d),Machine Learning
6,5955781475,Software Engineer - Computer Vision & Künstliche Intelligenz (m/w/d),Software engineers
7,5625824028,Backend Developer (m/w/d) – IMES Industrie-Software / Microservices,Backend Developer
8,5938631492,"Machine Learning Engineer (m/w/d) | Java, Python - 75.000€",Machine Learning
9,5959814190,.NET Software Engineer,Software engineers


In [22]:
output = Path("../data/evaluation_jobs.csv")

evaluation_jobs.to_csv(output, index=False)

print(f"Saved {len(evaluation_jobs)} jobs")

Saved 28 jobs


In [23]:
evaluation_jobs.to_csv(
    "../data/evaluation_jobs.csv",
    index=False
)

print(f"Saved {len(evaluation_jobs)} evaluation jobs")

Saved 28 evaluation jobs


In [ ]:
#keeping this code for new labels for the new candidate profiles
# labels = []

# for profile_id in candidate_profiles:
#     for _, job in evaluation_jobs.iterrows():
#         labels.append({
#             "profile_id": profile_id,
#             "job_id": str(job["id"]),
#             "job_title": job["title"],
#             "relevance": ""
#         })

# labels_df = pd.DataFrame(labels)

# labels_df.to_csv(
#     "../data/relevance_labels.csv",
#     index=False
# )

# print("Labels to assign:", len(labels_df))
# display(labels_df.head())
# print("Total judgments needed:", len(labels_df))



#Since we already finished labeling the relevance scores, we can load the labels from the CSV file
# Load existing labels instead of creating new ones

labels_df = pd.read_csv(
    "../data/relevance_labels.csv",
    dtype={"job_id": str}
)

print("Total judgments:", len(labels_df))
display(labels_df.head())



Total judgments: 140


,profile_id,candidate_profile,job_id,job_title,description,relevance
0,ml_engineer,"\n Junior machine learning engineer with Python, Pandas,\n scikit-learn, SQL, data preprocessing, model evaluation,\n and experience building predictive models.\n Looki...",5953648768,Cloud & AI Engineer Python (m/w/d) 100.000€,"Cloud & AI Engineer Python (m/w/d) Unbefristete Festanstellung | Frankfurt am Main | Gehalt bis 100.000 € Über das Unternehmen Für meinen Mandanten, ein führendes Beratungs- und Entwicklungsuntern...",1
1,ml_engineer,"\n Junior machine learning engineer with Python, Pandas,\n scikit-learn, SQL, data preprocessing, model evaluation,\n and experience building predictive models.\n Looki...",5943389711,Data Scientist (m/w/d),Gesundheit ist genau Dein Ding? Unseres auch! WEFRA LIFE sucht eine*n Data Scientist (w/m/d) Die treibende Kraft im Gesundheitsmarkt kann man nicht alleine sein. Dafür kommt es auf jeden Einzelnen...,1
2,ml_engineer,"\n Junior machine learning engineer with Python, Pandas,\n scikit-learn, SQL, data preprocessing, model evaluation,\n and experience building predictive models.\n Looki...",5956672032,"(Junior) Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur",Aufgaben - Als (Junior) Data Scientist (m/w/d) entwickelst und implementierst du eigenständig Datenmodelle und -algorithmen für unsere kundenspezifischen Lösungen. Dabei identifizierst und analys...,2
3,ml_engineer,"\n Junior machine learning engineer with Python, Pandas,\n scikit-learn, SQL, data preprocessing, model evaluation,\n and experience building predictive models.\n Looki...",5939347586,AI Engineer (m/w/d) - Implementation & Consulting,"html AI Engineer (m/w/d) - Implementation & Consulting Keine PowerPoint-Beratung: Wir bauen KI-Lösungen, die im Alltag laufen. Von Wissensplattformen über Agenten-Workflows bis zu automatisierten ...",1
4,ml_engineer,"\n Junior machine learning engineer with Python, Pandas,\n scikit-learn, SQL, data preprocessing, model evaluation,\n and experience building predictive models.\n Looki...",5892995304,Computer Vision Intern – Machine Learning Operations (m/f/d),"To strengthen our European headquarter in Frankfurt am Main, we are currently looking for a Computer Vision Intern – Machine Learning Operations (m/f/d) About us MOBIS Parts Europe N.V. is the Eur...",1


In [27]:
# Load our evaluation data
evaluation_jobs = pd.read_csv(
    "../data/evaluation_jobs.csv",
    dtype={"id": str}
)

labels_df = pd.read_csv(
    "../data/relevance_labels.csv",
    dtype={"job_id": str}
)

# Combine labels with job descriptions
review_df = labels_df.merge(
    evaluation_jobs[["id", "description"]],
    left_on="job_id",
    right_on="id",
    how="left"
)

# Current position in the labeling process
current_index = 0




---
## 9. What the baseline tells you

In prose: how good is the baseline, which mistakes does it make, and what do you expect your solution to fix?

The TF-IDF baseline provides a simple and fast way to match candidate profiles with job descriptions based on shared keywords. The initial results were promising, as the model recommended several relevant Python, AI, and software engineering positions.

However, the baseline has limitations. It relies on word overlap rather than understanding the meaning of the text. As a result, it may miss relevant jobs that use different terminology or recommend positions that share technical keywords but require different skills or experience levels.

I expect Sentence Transformers to improve the recommendations by capturing semantic relationships between candidate experience and job requirements. This should help the system identify suitable jobs even when the candidate and job descriptions use different words.